In [ ]:
# Install Pytorch & other libraries
%pip install torch tensorboard

# Install Transformers
%pip install transformers
%pip install --upgrade transformers
# Install Hugging Face libraries
%pip install datasets accelerate evaluate bitsandbytes trl peft protobuf sentencepiece

In [ ]:
import re
from random import randint

import torch
from datasets import load_dataset
from peft import LoraConfig
from transformers import pipeline, GenerationConfig
from trl import SFTConfig, SFTTrainer

In [ ]:
# Login into Hugging Face Hub
from huggingface_hub import login
login()

## Load the fine-tuning dataset



In [ ]:
dataset = load_dataset("Meriem-DH/marine-dataset-qa")

def format_sft(sample):
    return {
        "messages": [
            {"role": "user",      "content": sample["instruction"]},
            {"role": "assistant", "content": sample["response"]},
        ]
    }

dataset = dataset.map(format_sft, remove_columns=dataset["train"].column_names)

print(dataset["train"][0])
print(dataset["test"][0])

{'messages': [{'content': 'What is ice?', 'role': 'user'}, {'content': 'Ice is water that is frozen into a solid state, typically forming at or below temperatures of 0 °C, 32 °F, or 273.15 K.', 'role': 'assistant'}]}
{'messages': [{'content': 'What is the name of the French inter-ministerial project that aims to protect and sustainably manage coral reefs in the South Pacific?', 'role': 'user'}, {'content': 'Coral Reef Initiative for the South Pacific (CRISP)', 'role': 'assistant'}]}


## Fine-tune  using TRL and the SFTTrainer


In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

torch_dtype = torch.float16

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch_dtype,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch_dtype,
    )
)

# freeze base model
for name, param in model.named_parameters():
    if "lora" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False


model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)

tokenizer = AutoTokenizer.from_pretrained(model_id)


In [ ]:
peft_config = LoraConfig(
    lora_alpha=8, 
    lora_dropout=0.05,
    r=8,
    bias="none",
    target_modules="all-linear",
    task_type="CAUSAL_LM",
    ensure_weight_tying=True,
)

In [ ]:
# define hyperparameter of the trainer
args = SFTConfig(
    output_dir="qwen-sft",         # directory to save and repository id
    max_length=512,                         # max length for model and packing of the dataset
    num_train_epochs=3,                     # number of training epochs
    per_device_train_batch_size=1,          # batch size per device during training
    optim="adamw_torch_fused",
    logging_steps=10,                       # log every 10 steps
    save_strategy="epoch",                  # save checkpoint every epoch
    eval_strategy="epoch",                  # evaluate checkpoint every epoch
    learning_rate=2e-5,                     # learning rate
    fp16=False,   # half-precision 16-bit float, reduces memory and speeds up training, widely supported on GPUs
    bf16=True,   # 16-bit Brain float with larger numeric range but less fraction precision, only fully supported on certain GPUs (A100/H100)
    max_grad_norm=0.3,
    lr_scheduler_type="cosine",           # use constant learning rate scheduler
    push_to_hub=False,                        # push model to hub
    #hub_model_id="gemma-4-ocean-cpt",
    report_to="tensorboard",                # report metrics to tensorboard
    dataset_kwargs={
        "add_special_tokens": False, # Template with special tokens
        "append_concat_token": True, # Add EOS token as separator token between examples
    }
)

In [ ]:
# Create Trainer object
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    peft_config=peft_config,
    processing_class=tokenizer,
)

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1209: UserWarning: You have requested `ensure_weight_tying`, but no tied modules are added in `modules_to_save`
  warnings.warn(


In [ ]:
# Start training, the model will be automatically saved to the Hub and the output directory
trainer.train()

In [15]:
# Optional : save the final model again to the Hugging Face Hub
trainer.save_model()

## Test Model Inference

In [ ]:
config = GenerationConfig.from_pretrained(model_id)
config.max_new_tokens = 256

# Load the model and tokenizer into the pipeline
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

# Load a random sample from the test dataset
rand_idx = randint(0, len(dataset["test"]))
test_sample = dataset["test"][rand_idx]

# Convert as test example into a prompt with the Gemma template
prompt = pipe.tokenizer.apply_chat_template(
    test_sample["messages"][:1],
    tokenize=False,
    add_generation_prompt=True
)

print(type(prompt))
print(prompt)

# Generate our SQL query.
outputs = pipe(prompt, generation_config=config)
print(outputs[0]["generated_text"])

In [ ]:
# save and push the model
trainer.save_model("ocean-AI-sft")
tokenizer.save_pretrained("ocean-AI-sft")

trainer.push_to_hub("Meriem-DH/ocean-AI-sft")
tokenizer.push_to_hub("Meriem-DH/ocean-AI-sft")